In [ ]:
import os
import numpy as np
from scipy import signal
import quantities as pq
import math 
import neo
import json
from pathlib import Path
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, Cursor
import pickle
import sys 
from datetime import datetime
import shutil
from ast import literal_eval
from scipy.signal import find_peaks
from scipy.stats import zscore
from scipy import stats
from itertools import groupby
from IPython.display import display
from scipy.interpolate import griddata
import numpy as np
from scipy.signal import butter, filtfilt, hilbert
import matplotlib.pyplot as plt
from scipy import interpolate
import numpy.matlib
from sklearn.decomposition import PCA
from scipy import stats
import numpy as np
import pickle
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import resample
from scipy.signal import resample_poly
from math import gcd
import warnings
import numpy as np
from scipy.stats import zscore
import matplotlib
BuRd = matplotlib.cm.get_cmap('RdBu_r')
warnings.filterwarnings("ignore")
import sys
class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()


minian_path = os.path.join(os.path.abspath('..'),'minian')
print("The folder used for minian procedures is : {}".format(minian_path))
sys.path.append(minian_path)

from minian.utilities import (
    TaskAnnotation,
    get_optimal_chk,
    load_videos,
    open_minian,
    save_minian,
)
#######################################################################################
                                # Define functions #
#######################################################################################

def resample_matrix(data, orig_rate, target_rate, axis=0):
    if orig_rate == target_rate:
        return data.copy()
    # Compute integer up/down factors using GCD
    up = int(target_rate)
    down = int(orig_rate)
    factor = gcd(up, down)
    up //= factor
    down //= factor
    return resample_poly(data, up=up, down=down, axis=axis)


def bin_sum_fractional(x, Fs1, Fs2):
    N = len(x)
    bin_width = Fs1 / Fs2
    y = []
    start = 0
    while start < N:
        end = start + bin_width
        i_start = int(np.floor(start))
        i_end = int(np.floor(end))
        if i_end >= N:
            i_end = N - 1
        total = 0.0
        total += x[i_start] * (1 - (start - i_start))
        for i in range(i_start + 1, i_end):
            total += x[i]
        if i_end > i_start:
            total += x[i_end] * (end - i_end)
        y.append(total)
        start = end
    return np.array(y)

def marcenkopastur(significance):
    nbins = significance.nbins
    nneurons = significance.nneurons
    tracywidom = significance.tracywidom
    q = float(nbins)/float(nneurons)
    lambdaMax = pow((1+np.sqrt(1/q)),2)
    lambdaMax += tracywidom*pow(nneurons,-2./3)
    return lambdaMax

def getlambdacontrol(zactmat_):
    significance_ = PCA()
    significance_.fit(zactmat_.T)
    lambdamax_ = np.max(significance_.explained_variance_)
    return lambdamax_

def binshuffling(zactmat,significance):
    np.random.seed()
    lambdamax_ = np.zeros(significance.nshu)
    for shui in range(significance.nshu):
        zactmat_ = np.copy(zactmat)
        for (neuroni,activity) in enumerate(zactmat_):
            randomorder = np.argsort(np.random.rand(significance.nbins))
            zactmat_[neuroni,:] = activity[randomorder]
        lambdamax_[shui] = getlambdacontrol(zactmat_)
    lambdaMax = np.percentile(lambdamax_,significance.percentile)
    return lambdaMax

def circshuffling(zactmat,significance):
    np.random.seed()
    lambdamax_ = np.zeros(significance.nshu)
    for shui in range(significance.nshu):
        zactmat_ = np.copy(zactmat)
        for (neuroni,activity) in enumerate(zactmat_):
            cut = int(np.random.randint(significance.nbins*2))
            zactmat_[neuroni,:] = np.roll(activity,cut)
        lambdamax_[shui] = getlambdacontrol(zactmat_)
    lambdaMax = np.percentile(lambdamax_,significance.percentile)
    return lambdaMax

def runSignificance(zactmat,significance):
    if significance.nullhyp == 'mp':
        lambdaMax = marcenkopastur(significance)
    elif significance.nullhyp == 'bin':
        lambdaMax = binshuffling(zactmat,significance)
    elif significance.nullhyp == 'circ':
        lambdaMax = circshuffling(zactmat,significance)
    else:
        print('ERROR !')
        print('    nyll hypothesis method '+str(nullhyp)+' not understood')
        significance.nassemblies = np.nan
    nassemblies = np.sum(significance.explained_variance_>lambdaMax)
    significance.nassemblies = nassemblies
    return significance, lambdaMax

def extractPatterns(actmat,significance,method):
    nassemblies = significance.nassemblies
    if method == 'pca':
        idxs = np.argsort(-significance.explained_variance_)[0:nassemblies]
        patterns = significance.components_[idxs,:]
    elif method == 'ica':
        from sklearn.decomposition import FastICA
        ica = FastICA(n_components=nassemblies, max_iter=1000)
        ica.fit(actmat.T)
        patterns = ica.components_
    else:
        print('ERROR !')
        print('    assembly extraction method '+str(method)+' not understood')
        patterns = np.nan
    if patterns is not np.nan:
        patterns = patterns.reshape(nassemblies,-1)
        norms = np.linalg.norm(patterns,axis=1)
        patterns /= np.matlib.repmat(norms,np.size(patterns,1),1).T
    return patterns

def runPatterns(actmat, method='ica', nullhyp = 'mp', nshu = 1000, percentile = 99, tracywidom = False):
    nneurons = np.size(actmat,0)
    nbins = np.size(actmat,1)
    silentneurons = np.var(actmat,axis=1)==0
    actmat_ = actmat[~silentneurons,:]
    zactmat_ = stats.zscore(actmat_,axis=1)
    significance = PCA()
    significance.fit(zactmat_.T)
    significance.nneurons = nneurons
    significance.nbins = nbins
    significance.nshu = nshu
    significance.percentile = percentile
    significance.tracywidom = tracywidom
    significance.nullhyp = nullhyp
    significance, lambdaMax = runSignificance(zactmat_,significance)
    if np.isnan(significance.nassemblies):
        patterns = []
        zactmat = []
        significance = []
        #return
    if significance.nassemblies<1:
        print('WARNING 1!')
        print('    no assembly detected!')
        patterns = []
        zactmat = []
        significance = []
    else:
        patterns_ = extractPatterns(zactmat_,significance,method)
        if patterns_ is np.nan:
            patterns = []
            zactmat = []
            significance = []
            #return
        patterns = np.zeros((np.size(patterns_,0),nneurons))
        patterns[:,~silentneurons] = patterns_
        zactmat = np.copy(actmat)
        zactmat[~silentneurons,:] = zactmat_
    return patterns,significance,zactmat, lambdaMax

def computeAssemblyActivity(patterns,zactmat,zerodiag = True):
    if len(patterns) == 0:
        print('WARNING 2!')
        print('    no assembly detecded!')
        assemblyAct = []
    else:
        nassemblies = len(patterns)
        nbins = np.size(zactmat,1)
        assemblyAct = np.zeros((nassemblies,nbins))
        for (assemblyi,pattern) in enumerate(patterns):
            projMat = np.outer(pattern,pattern)
            projMat -= zerodiag*np.diag(np.diag(projMat))
            for bini in range(nbins):
                assemblyAct[assemblyi,bini] = np.dot(np.dot(zactmat[:,bini],projMat),zactmat[:,bini])
    return assemblyAct


In [ ]:
minian_path = os.path.join(os.path.abspath('..'),'minian')
print("The folder used for minian procedures is : {}".format(minian_path))
sys.path.append(minian_path)

In [ ]:
from minian.utilities import (
    TaskAnnotation,
    get_optimal_chk,
    load_videos,
    open_minian,
    open_minian_mf,
    save_minian,
)

Stop on wanted cell assembly properties

In [ ]:
DrugExperiment=0 # =1 if CGP Experiment // DrugExperiment=0 if Baseline Experiment

AnalysisID='' 

saveexcel=0

local = True
if local:
    dir = "//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_data/L1NDNF_mice/"
else: 
    dir = "/mnt/data/AurelieB_other/"


mapp = {1: 'AW',  2: 'QW', 3: 'NREM',  4: 'IS', 5: 'REM',  6: 'undefined'}
drugs=['baseline']

CellAssembly_dict={}
CellAssembly_members={}
for dpath in Path(dir).glob('**/mappingsAB.pkl'):

    mappfile = open(dpath.parents[0]/ f'mappingsAB.pkl', 'rb')
    mapping = pickle.load(mappfile)
    mapping_sess = mapping['session']   

    centroidfile = open(dpath.parents[0]/ f'centsAB.pkl', 'rb')
    centroids = pickle.load(centroidfile)
    centroids_sess = centroids['session']   

    mice = dpath.parents[1].parts[-1]
    NeuronType = dpath.parents[2].parts[-1]
    
    print(f"####################################################################################")
    print(f"################################### {mice} ####################################")
    print(f"####################################################################################")

    nb_minian_total=0
    dict_Path={}
    dict_Calcium = {}
    dict_Deconv = {}
    dict_Scoring = {}
    dict_Stamps = {}
    dict_TodropFile = {}
    dict_StampsMiniscope = {}

    CellAssembly_patterns=pd.DataFrame()  
    CellAssembly_members[mice]={}

    previousEndTime=0
    InitialStartTime=0

    assembly_nb=0

    minian_folders = [f for f in dpath.parents[0].rglob('minian') if f.is_dir()]

    for minianpath in minian_folders: # for each minian folders found in this mouse 

        if 1==1: # any(p in all_expe_types for p in minianpath.parts): # have to be to the expe_types

            if any(minianpath.parents[1].glob('*V4_Miniscope/timeStamps.csv')): 
                session_path=minianpath.parents[1]        
                tsmini= pd.read_csv(list(session_path.glob('*V4_Miniscope/timeStamps.csv'))[0])['Time Stamp (ms)']
                V4subfolder=False               
                if any(session_path.parent.rglob('OpenEphys/EphyViewer_ManualScor5s_6Stages.csv')):
                    session_type=minianpath.parents[3].name
                    session_date=minianpath.parents[4].name
                    session_time=minianpath.parents[1].name
                else:
                    session_type=minianpath.parents[2].name
                    session_date=minianpath.parents[3].name
                    session_time=minianpath.parents[1].name
                session=session_time        
            elif any(minianpath.parents[2].glob('*V4_Miniscope/timeStamps.csv')):
                session_path=minianpath.parents[2]      
                tsmini= pd.read_csv(list(session_path.glob('*V4_Miniscope/timeStamps.csv'))[0])['Time Stamp (ms)']                      
                V4subfolder=True
                session_type=minianpath.parents[4].name
                session_date=minianpath.parents[5].name
                session_time=minianpath.parents[0].name
                session=session_time

            if session_type == 'Cheeseboard':

                print(f"Processing {session_type} session: {session} on the {session_date}, subfolders = {V4subfolder}")

                minian_ds = open_minian(minianpath)
                dict_Calcium[session] = minian_ds['C'] # calcium traces 
                dict_Deconv[session] = minian_ds['S'] # estimated spikes deconvolved activity
                
                try: 
                    try: 
                        with open(minianpath.parent / f'TodropFileAB.json', 'r') as f:
                            unit_to_drop = json.load(f)
                            dict_TodropFile[session]  = unit_to_drop
                    except: 
                        with open(minianpath/ f'TodropFileAB.json', 'r') as f:
                            unit_to_drop = json.load(f)
                            dict_TodropFile[session]  = unit_to_drop
                    nb_minian_total+=1
                except:
                    print(' !!!! Minian session not validated !!!!')
                    continue
                print(session_path)

                
                # Start time & freq miniscope
                
                dict_StampsMiniscope[session]  = pd.read_csv(list(session_path.glob('*V4_Miniscope/timeStamps.csv'))[0])
                tsmini=dict_StampsMiniscope[session]['Time Stamp (ms)']
                minian_freq=round(1/np.mean(np.diff(np.array(tsmini)/1000)))  

                freqLFP=1000    
                numbdropfr= 0         
                
                nb_minian_total+=1

                #######################################################################################
                # Distribute Ca2+ intensity & spikes to vigilance states for each sessions #
                #######################################################################################
                    
                # Adjust the StartTime if subsessions

                dict_Stamps[session]  = pd.read_excel(session_path/ f'SynchroFile.xlsx')     
                StartTime = list(dict_Stamps[session][0])[0] 
                if math.isnan(StartTime):   
                    StartTime = 0 # No OpenEphys file found, so Miniscope start relative to OpenEphys start equal 0

                if InitialStartTime==0:
                    InitialStartTime=StartTime    
                    firstframe=0
                    StartTimeMiniscope=0 # start time of miniscope rec of that subsesssions relative to the start of the mniscope recording
                else:
                    if StartTime == InitialStartTime: # just a subsession
                        StartTime = previousEndTime + 1/minian_freq #  +1 frame in seconds
                        StartTimeMiniscope= StartTime-InitialStartTime
                    else:  
                        InitialStartTime=StartTime # this is a new session
                        firstframe=0
                        StartTimeMiniscope=0 
                        

                # Remove bad units from recordings

                C=dict_Calcium[session]
                D=dict_Deconv[session] 
                unit_to_drop=dict_TodropFile[session]    

                C_sel=C.drop_sel(unit_id=unit_to_drop)
                D_sel=D.drop_sel(unit_id=unit_to_drop)

                CalciumSub = pd.DataFrame(C_sel, index=C_sel['unit_id'])
                DeconvSub = pd.DataFrame(D_sel, index=D_sel['unit_id'])

                indexMappList=mapping_sess[session]
                kept_uniq_unit_List=[]
                for unit in CalciumSub.index:
                    indexMapp = np.where(indexMappList == unit)[0]
                    kept_uniq_unit_List.append(f"{mice}{str(indexMapp).replace('[','').replace(']','')}")

                nb_unit=len(CalciumSub)
                if nb_unit==0:
                    print(f'no cells kept in the session: {session}')
                    continue  # next iteration
                

                # Realigned the traces to the recorded timestamps 

                timestamps =  np.array(tsmini[firstframe:firstframe+len(CalciumSub.T)])/freqLFP
                new_timestamps= np.arange(timestamps[0], timestamps[-1], 1/minian_freq)
                
                Calcium = pd.DataFrame(index=CalciumSub.index, columns=new_timestamps)
                for feature in CalciumSub.index:
                    interpolator = interpolate.interp1d(timestamps, CalciumSub.loc[feature], kind='linear')
                    Calcium.loc[feature] = interpolator(new_timestamps)
                Carray=Calcium.values.T.astype(float) # Calcium activity

                Deconv = pd.DataFrame(index=DeconvSub.index, columns=new_timestamps)
                for feature in DeconvSub.index:
                    interpolator = interpolate.interp1d(timestamps, DeconvSub.loc[feature], kind='linear')
                    Deconv.loc[feature] = interpolator(new_timestamps)
                Darray=Deconv.values.T.astype(float) # Deconvolved activity

                Sarray= np.zeros((np.shape(Darray)[0], np.shape(Darray)[1])) # Spike activity
                for i in np.arange(np.shape(Darray)[1]):
                    Darray_unit =Darray[:,i]
                    peaks, _ = find_peaks(Darray_unit)
                    Sarray_unit=np.zeros(len(Darray_unit))
                    Sarray_unit[peaks]=1   
                    Sarray[:,i]=Sarray_unit
                Spike= pd.DataFrame(Sarray.T, index=CalciumSub.index, columns=new_timestamps)

                firstframe+=len(CalciumSub.T)
                rec_dur=len(CalciumSub.T)
                rec_dur_sec= timestamps[-1]- timestamps[0] 
                EndTime = StartTime + rec_dur_sec # (upd_rec_dur/minian_freq) # in seconds
                previousEndTime=EndTime 

                print(session, ': starts at', round(StartTime,1), 's & ends at', round(EndTime,1), 's (', round(rec_dur_sec,1), 's duration, ', numbdropfr, 'dropped frames, minian frequency =', minian_freq, 'Hz, experiment type = ', session_type, ')...') 
                sentence1= f"... kept values = {kept_uniq_unit_List}"
                print(sentence1)

                # Define cell assemblies            
                target_rate =20 # 20 Hz == 50 ms bins 5 Hz = 200msbins
                Array_bin = resample_matrix(Carray, orig_rate=minian_freq, target_rate=target_rate)
                #Array_bin = resample_matrix(Darray, orig_rate=minian_freq, target_rate=target_rate)
                #Array_bin = bin_sum_fractional(Sarray, minian_freq, target_rate) 


                # Plot calcium signal in bins over time
                plt.figure(figsize=(20, 8))
                plt.imshow(Array_bin.T, aspect='auto', vmin=0, vmax=1)
                plt.yticks(ticks=range(len(kept_uniq_unit_List)), labels=kept_uniq_unit_List, fontsize=7)
                plt.title('Binned calcium signals')
                plt.savefig("//10.69.168.1/crnldata/forgetting/Théa/MiniscopeOE_analysis/plot_bins.svg", bbox_inches="tight")
                plt.show()

                # Identify cell assembly
                patterns,significance,zactmat,lambdaMax= runPatterns(Array_bin.T, method='ica', nullhyp = 'mp', nshu = 1000, percentile = 99, tracywidom = False)       


                # reverse negative patterns

                for ass in np.arange(np.shape(patterns)[0]):
                    if np.mean(patterns[ass]) <0:
                        patterns[ass]=-patterns[ass]
                
                all_patterns = pd.DataFrame({ass: patterns[ass].tolist() for ass in np.arange(np.shape(patterns)[0])}, index=kept_uniq_unit_List).add_prefix(f"{session_time}_CellAss")
                CellAssembly_patterns = CellAssembly_patterns.join(all_patterns, how="outer") if  not CellAssembly_patterns.empty else all_patterns 

                if len(patterns)>0:
                    patterns_th = patterns.copy()
                    for ass in np.arange(np.shape(patterns)[0]) :
                        thresh =  np.mean(patterns_th[ass]) +  2 * np.std(patterns_th[ass])
                        patterns_th[ass][(patterns_th[ass])<thresh]=np.nan
                        #patterns_th[ass][(patterns_th[ass])<thresh]=np.nan # only positive weight
                        non_nan_indices =  np.where(~np.isnan(patterns_th[ass]))[0]
                        assembly_ID = all_patterns.columns.tolist()[ass]
                        cells_in_assembly=np.array(kept_uniq_unit_List)[non_nan_indices].tolist()
                        CellAssembly_members[mice][assembly_ID]=cells_in_assembly



                    #Plot principal components
                    df = pd.DataFrame({"value": significance.explained_variance_})
                    x = np.arange(len(df))
                    y = df["value"].values
                    fig, ax = plt.subplots(figsize=(max(5, len(df)*0.2), 2))
                    ax.vlines(x, ymin=0, ymax=y, linewidth=2,  color='k')
                    # heads (markers)
                    ax.scatter(x, y, s=50, zorder=3, edgecolors=None, color='k')
                    ax.axhline(y=lambdaMax, color='r', linestyle='--', label='Mean')
                    ax.set_xticks(x)
                    ax.set_xticklabels( df.index)
                    ax.set_ylim(0, max(y)*1.2)
                    ax.set_ylabel("Value")
                    ax.set_title(f"{session_date}, {session_time}, {mice}")
                    plt.tight_layout()
                    #plt.savefig(f'C:/Users/Manip2/Documents/PCA{session_date}_{session_time}_{mice}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
                    plt.savefig("//10.69.168.1/crnldata/forgetting/Théa/MiniscopeOE_analysis/plot_components.svg", bbox_inches="tight")
                    plt.show()


                    # Plot template of each cell assemblies
                    
                    n_ass = np.shape(patterns)[0]
                    if n_ass > 0:
                        cols = 3  # Number of columns in subplot grid; adjust as needed
                        rows = math.ceil(n_ass / cols)
                        fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 5))
                        axes = axes.flatten()  # Flatten to 1D array for easy indexing
                        
                        for ass in np.arange(n_ass):
                            template = np.add.outer(patterns[ass], patterns[ass])      
                            template = np.nan_to_num(template, nan=0)
                            template = template - np.diag(np.diag(template))
                            
                            axes[ass].imshow(template, cmap= BuRd, vmin=-.5, vmax=.5)
                            axes[ass].set_title(f"{session_date}, {session_time}, {mice}, {ass}", fontsize=8)
                            axes[ass].set_xticks(np.arange(len(kept_uniq_unit_List)))
                            axes[ass].set_yticks(np.arange(len(kept_uniq_unit_List)))
                            axes[ass].set_xticklabels(kept_uniq_unit_List, rotation=45, fontsize=5,ha='right')
                            axes[ass].set_yticklabels(kept_uniq_unit_List, fontsize=5)

                        # Hide any unused subplots
                        for i in range(n_ass, len(axes)):
                            axes[i].set_visible(False)
                        
                        plt.tight_layout()
                        plt.savefig("//10.69.168.1/crnldata/forgetting/Théa/MiniscopeOE_analysis/plot_templates.svg", bbox_inches="tight")
                        plt.show()




                                        
                    
                    # Print significant cell in cell assemblies
                    patterns_th = patterns.copy()
                    for ass in np.arange(np.shape(patterns)[0]):
                        thresh = np.mean(patterns_th[ass]) + 2 * np.std(patterns_th[ass])

                        patterns_th[ass][(patterns_th[ass])<thresh]=np.nan
                        non_nan_indices = np.where(~np.isnan(patterns_th[ass]))[0] 
                        assembly_ID = all_patterns.columns.tolist()[ass] 
                        cells_in_assembly = np.array(kept_uniq_unit_List)[non_nan_indices].tolist()
                        print(f"Cells in assembly {assembly_ID}: {cells_in_assembly}")

                        
                        
                        

                    # Plot cell weight for each assemblies
                    cols = 2  # Number of columns in subplot grid; adjust as needed
                    rows = math.ceil(n_ass / cols)
                    fig, axes = plt.subplots(rows, cols, figsize=(max(15, n_ass * 4), 4 * rows))
                    axes = axes.flatten()  # Flatten to 1D array for easy indexing                    
                    for ass in np.arange(n_ass):
                        thresh = np.mean((patterns[ass])) + 2 * np.std(patterns[ass])

                        df = pd.DataFrame({
                            "neuron": kept_uniq_unit_List,
                            "value": patterns[ass]
                        })
                        x = np.arange(len(df))
                        y = df["value"].values                        
                        axes[ass].vlines(x, ymin=0, ymax=y, linewidth=2, color='k')
                        axes[ass].scatter(x, y, s=50, zorder=3, edgecolors=None, color='k')
                        axes[ass].axhline(y=thresh, color='r', linestyle='--')
                        axes[ass].axhline(y=-thresh, color='r', linestyle='--')
                        axes[ass].axhline(y=0, color='k')
                        axes[ass].set_xticks(x)
                        axes[ass].set_xticklabels(df['neuron'], rotation=45, fontsize=7, ha='right')
                        axes[ass].set_ylabel("Value")
                        axes[ass].set_title(f"{session_date}, {session_time}, {mice}, {ass}")                    
                    for i in range(n_ass, len(axes)):
                        axes[i].set_visible(False)                    
                    plt.tight_layout()
                    plt.savefig("//10.69.168.1/crnldata/forgetting/Théa/MiniscopeOE_analysis/plot_cell_weights.svg", bbox_inches="tight")
                    plt.show()



                    print('______________________________________________________________________________________________________________________________________________________________________________________________________________________________')

In [ ]:
DrugExperiment=0 # =1 if CGP Experiment // DrugExperiment=0 if Baseline Experiment
 
AnalysisID='' 
 
saveexcel=0
 
local = True
if local:
    dir = "//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_data/L1NDNF_mice/"
else: 
    dir = "/mnt/data/AurelieB_other/"
 
 
mapp = {1: 'AW',  2: 'QW', 3: 'NREM',  4: 'IS', 5: 'REM',  6: 'undefined'}
drugs=['baseline']
 
CellAssembly_dict={}
CellAssembly_members={}
for dpath in Path(dir).glob('**/mappingsAB.pkl'):
 
    mappfile = open(dpath.parents[0]/ f'mappingsAB.pkl', 'rb')
    mapping = pickle.load(mappfile)
    mapping_sess = mapping['session']   
 
    centroidfile = open(dpath.parents[0]/ f'centsAB.pkl', 'rb')
    centroids = pickle.load(centroidfile)
    centroids_sess = centroids['session']   
 
    mice = dpath.parents[1].parts[-1]
    NeuronType = dpath.parents[2].parts[-1]
    
    print(f"####################################################################################")
    print(f"################################### {mice} ####################################")
    print(f"####################################################################################")
 
    nb_minian_total=0
    dict_Path={}
    dict_Calcium = {}
    dict_Deconv = {}
    dict_Scoring = {}
    dict_Stamps = {}
    dict_TodropFile = {}
    dict_StampsMiniscope = {}
 
    CellAssembly_patterns=pd.DataFrame()  
    CellAssembly_members[mice]={}
 
    previousEndTime=0
    InitialStartTime=0
 
    assembly_nb=0
 
    minian_folders = [f for f in dpath.parents[0].rglob('minian') if f.is_dir()]
 
    for minianpath in minian_folders: # for each minian folders found in this mouse 
 
        if 1==1: # any(p in all_expe_types for p in minianpath.parts): # have to be to the expe_types
 
            if any(minianpath.parents[1].glob('*V4_Miniscope/timeStamps.csv')): 
                session_path=minianpath.parents[1]        
                tsmini= pd.read_csv(list(session_path.glob('*V4_Miniscope/timeStamps.csv'))[0])['Time Stamp (ms)']
                V4subfolder=False               
                if any(session_path.parent.rglob('OpenEphys/EphyViewer_ManualScor5s_6Stages.csv')):
                    session_type=minianpath.parents[3].name
                    session_date=minianpath.parents[4].name
                    session_time=minianpath.parents[1].name
                else:
                    session_type=minianpath.parents[2].name
                    session_date=minianpath.parents[3].name
                    session_time=minianpath.parents[1].name
                session=session_time        
            elif any(minianpath.parents[2].glob('*V4_Miniscope/timeStamps.csv')):
                session_path=minianpath.parents[2]      
                tsmini= pd.read_csv(list(session_path.glob('*V4_Miniscope/timeStamps.csv'))[0])['Time Stamp (ms)']                      
                V4subfolder=True
                session_type=minianpath.parents[4].name
                session_date=minianpath.parents[5].name
                session_time=minianpath.parents[0].name
                session=session_time
 
            if session_type == 'Cheeseboard':
 
                print(f"Processing {session_type} session: {session} on the {session_date}, subfolders = {V4subfolder}")
 
                minian_ds = open_minian(minianpath)
                dict_Calcium[session] = minian_ds['C'] # calcium traces 
                dict_Deconv[session] = minian_ds['S'] # estimated spikes deconvolved activity
                
                try: 
                    try: 
                        with open(minianpath.parent / f'TodropFileAB.json', 'r') as f:
                            unit_to_drop = json.load(f)
                            dict_TodropFile[session]  = unit_to_drop
                    except: 
                        with open(minianpath/ f'TodropFileAB.json', 'r') as f:
                            unit_to_drop = json.load(f)
                            dict_TodropFile[session]  = unit_to_drop
                    nb_minian_total+=1
                except:
                    print(' !!!! Minian session not validated !!!!')
                    continue
                print(session_path)
 
                
                # Start time & freq miniscope
                
                dict_StampsMiniscope[session]  = pd.read_csv(list(session_path.glob('*V4_Miniscope/timeStamps.csv'))[0])
                tsmini=dict_StampsMiniscope[session]['Time Stamp (ms)']
                minian_freq=round(1/np.mean(np.diff(np.array(tsmini)/1000)))  
 
                freqLFP=1000    
                numbdropfr= 0         
                
                nb_minian_total+=1
 
                #######################################################################################
                # Distribute Ca2+ intensity & spikes to vigilance states for each sessions #
                #######################################################################################
                    
                # Adjust the StartTime if subsessions
 
                dict_Stamps[session]  = pd.read_excel(session_path/ f'SynchroFile.xlsx')     
                StartTime = list(dict_Stamps[session][0])[0] 
                if math.isnan(StartTime):   
                    StartTime = 0 # No OpenEphys file found, so Miniscope start relative to OpenEphys start equal 0
 
                if InitialStartTime==0:
                    InitialStartTime=StartTime    
                    firstframe=0
                    StartTimeMiniscope=0 # start time of miniscope rec of that subsesssions relative to the start of the mniscope recording
                else:
                    if StartTime == InitialStartTime: # just a subsession
                        StartTime = previousEndTime + 1/minian_freq #  +1 frame in seconds
                        StartTimeMiniscope= StartTime-InitialStartTime
                    else:  
                        InitialStartTime=StartTime # this is a new session
                        firstframe=0
                        StartTimeMiniscope=0 
                        
 
                # Remove bad units from recordings
 
                C=dict_Calcium[session]
                D=dict_Deconv[session] 
                unit_to_drop=dict_TodropFile[session]    
 
                C_sel=C.drop_sel(unit_id=unit_to_drop)
                D_sel=D.drop_sel(unit_id=unit_to_drop)
 
                CalciumSub = pd.DataFrame(C_sel, index=C_sel['unit_id'])
                DeconvSub = pd.DataFrame(D_sel, index=D_sel['unit_id'])
 
                indexMappList=mapping_sess[session]
                kept_uniq_unit_List=[]
                for unit in CalciumSub.index:
                    indexMapp = np.where(indexMappList == unit)[0]
                    kept_uniq_unit_List.append(f"{mice}{str(indexMapp).replace('[','').replace(']','')}")
 
                nb_unit=len(CalciumSub)
                if nb_unit==0:
                    print(f'no cells kept in the session: {session}')
                    continue  # next iteration
                
 
                # Realigned the traces to the recorded timestamps 
 
                timestamps =  np.array(tsmini[firstframe:firstframe+len(CalciumSub.T)])/freqLFP
                new_timestamps= np.arange(timestamps[0], timestamps[-1], 1/minian_freq)
                
                Calcium = pd.DataFrame(index=CalciumSub.index, columns=new_timestamps)
                for feature in CalciumSub.index:
                    interpolator = interpolate.interp1d(timestamps, CalciumSub.loc[feature], kind='linear')
                    Calcium.loc[feature] = interpolator(new_timestamps)
                Carray=Calcium.values.T.astype(float) # Calcium activity
 
                Deconv = pd.DataFrame(index=DeconvSub.index, columns=new_timestamps)
                for feature in DeconvSub.index:
                    interpolator = interpolate.interp1d(timestamps, DeconvSub.loc[feature], kind='linear')
                    Deconv.loc[feature] = interpolator(new_timestamps)
                Darray=Deconv.values.T.astype(float) # Deconvolved activity
 
                Sarray= np.zeros((np.shape(Darray)[0], np.shape(Darray)[1])) # Spike activity
                for i in np.arange(np.shape(Darray)[1]):
                    Darray_unit =Darray[:,i]
                    peaks, _ = find_peaks(Darray_unit)
                    Sarray_unit=np.zeros(len(Darray_unit))
                    Sarray_unit[peaks]=1   
                    Sarray[:,i]=Sarray_unit
                Spike= pd.DataFrame(Sarray.T, index=CalciumSub.index, columns=new_timestamps)
 
                firstframe+=len(CalciumSub.T)
                rec_dur=len(CalciumSub.T)
                rec_dur_sec= timestamps[-1]- timestamps[0] 
                EndTime = StartTime + rec_dur_sec # (upd_rec_dur/minian_freq) # in seconds
                previousEndTime=EndTime 
 
                print(session, ': starts at', round(StartTime,1), 's & ends at', round(EndTime,1), 's (', round(rec_dur_sec,1), 's duration, ', numbdropfr, 'dropped frames, minian frequency =', minian_freq, 'Hz, experiment type = ', session_type, ')...') 
                sentence1= f"... kept values = {kept_uniq_unit_List}"
                print(sentence1)
 
                # Define cell assemblies            
                target_rate =20 # 20 Hz == 50 ms bins 5 Hz = 200msbins
                Array_bin = resample_matrix(Carray, orig_rate=minian_freq, target_rate=target_rate)
                #Array_bin = resample_matrix(Darray, orig_rate=minian_freq, target_rate=target_rate)
                #Array_bin = bin_sum_fractional(Sarray, minian_freq, target_rate) 
 
 
                # Plot calcium signal in bins over time
                plt.figure(figsize=(20, 8))
                plt.imshow(Array_bin.T, aspect='auto', vmin=0, vmax=1)
                plt.yticks(ticks=range(len(kept_uniq_unit_List)), labels=kept_uniq_unit_List, fontsize=7)
                plt.title('Binned calcium signals')
                plt.savefig("//10.69.168.1/crnldata/forgetting/Théa/MiniscopeOE_analysis/plot_bins.svg", bbox_inches="tight")
                plt.show()
 
                # Identify cell assembly
                patterns,significance,zactmat,lambdaMax= runPatterns(Array_bin.T, method='ica', nullhyp = 'mp', nshu = 1000, percentile = 99, tracywidom = False)       
 
 
                # reverse negative patterns
 
                for ass in np.arange(np.shape(patterns)[0]):
                    if np.mean(patterns[ass]) <0:
                        patterns[ass]=-patterns[ass]
                
                all_patterns = pd.DataFrame({ass: patterns[ass].tolist() for ass in np.arange(np.shape(patterns)[0])}, index=kept_uniq_unit_List).add_prefix(f"{session_time}_CellAss")
                CellAssembly_patterns = CellAssembly_patterns.join(all_patterns, how="outer") if  not CellAssembly_patterns.empty else all_patterns 
 
                if len(patterns)>0:
                    patterns_th = patterns.copy()
                    for ass in np.arange(np.shape(patterns)[0]) :
                        thresh =  np.mean(patterns_th[ass]) +  2 * np.std(patterns_th[ass])
                        patterns_th[ass][(patterns_th[ass])<thresh]=np.nan
                        #patterns_th[ass][(patterns_th[ass])<thresh]=np.nan # only positive weight
                        non_nan_indices =  np.where(~np.isnan(patterns_th[ass]))[0]
                        assembly_ID = all_patterns.columns.tolist()[ass]
                        cells_in_assembly=np.array(kept_uniq_unit_List)[non_nan_indices].tolist()
                        CellAssembly_members[mice][assembly_ID]=cells_in_assembly
 
 
 
                    #Plot principal components
                    df = pd.DataFrame({"value": significance.explained_variance_})
                    x = np.arange(len(df))
                    y = df["value"].values
                    fig, ax = plt.subplots(figsize=(max(5, len(df)*0.2), 2))
                    ax.vlines(x, ymin=0, ymax=y, linewidth=2,  color='k')
                    # heads (markers)
                    ax.scatter(x, y, s=50, zorder=3, edgecolors=None, color='k')
                    ax.axhline(y=lambdaMax, color='r', linestyle='--', label='Mean')
                    ax.set_xticks(x)
                    ax.set_xticklabels( df.index)
                    ax.set_ylim(0, max(y)*1.2)
                    ax.set_ylabel("Value")
                    ax.set_title(f"{session_date}, {session_time}, {mice}")
                    plt.tight_layout()
                    #plt.savefig(f'C:/Users/Manip2/Documents/PCA{session_date}_{session_time}_{mice}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
                    plt.savefig("//10.69.168.1/crnldata/forgetting/Théa/MiniscopeOE_analysis/plot_components.svg", bbox_inches="tight")
                    plt.show()
 
 
                    # Plot template of each cell assemblies
                    
                    n_ass = np.shape(patterns)[0]
                    if n_ass > 0:
                        cols = 3  # Number of columns in subplot grid; adjust as needed
                        rows = math.ceil(n_ass / cols)
                        fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 5))
                        axes = axes.flatten()  # Flatten to 1D array for easy indexing
                        
                        for ass in np.arange(n_ass):
                            template = np.add.outer(patterns[ass], patterns[ass])      
                            template = np.nan_to_num(template, nan=0)
                            template = template - np.diag(np.diag(template))
                            
                            axes[ass].imshow(template, cmap= BuRd, vmin=-.5, vmax=.5)
                            axes[ass].set_title(f"{session_date}, {session_time}, {mice}, {ass}", fontsize=8)
                            axes[ass].set_xticks(np.arange(len(kept_uniq_unit_List)))
                            axes[ass].set_yticks(np.arange(len(kept_uniq_unit_List)))
                            axes[ass].set_xticklabels(kept_uniq_unit_List, rotation=45, fontsize=5,ha='right')
                            axes[ass].set_yticklabels(kept_uniq_unit_List, fontsize=5)
 
                        # Hide any unused subplots
                        for i in range(n_ass, len(axes)):
                            axes[i].set_visible(False)
                        
                        plt.tight_layout()
                        plt.savefig("//10.69.168.1/crnldata/forgetting/Théa/MiniscopeOE_analysis/plot_templates.svg", bbox_inches="tight")
                        plt.show()
 
 
                    # Print significant cell in cell assemblies
                    patterns_th = patterns.copy()
                    for ass in np.arange(np.shape(patterns)[0]):
                        thresh = np.mean(patterns_th[ass]) + 2 * np.std(patterns_th[ass])
 
                        patterns_th[ass][(patterns_th[ass])<thresh]=np.nan
                        non_nan_indices = np.where(~np.isnan(patterns_th[ass]))[0] 
                        assembly_ID = all_patterns.columns.tolist()[ass] 
                        cells_in_assembly = np.array(kept_uniq_unit_List)[non_nan_indices].tolist()
                        print(f"Cells in assembly {assembly_ID}: {cells_in_assembly}")
 
 
                    # Plot cell weight for each assemblies
                    cols = 2  # Number of columns in subplot grid; adjust as needed
                    rows = math.ceil(n_ass / cols)
                    fig, axes = plt.subplots(rows, cols, figsize=(max(15, n_ass * 4), 4 * rows))
                    axes = axes.flatten()  # Flatten to 1D array for easy indexing                    
                    for ass in np.arange(n_ass):
                        thresh = np.mean((patterns[ass])) + 2 * np.std(patterns[ass])
 
                        df = pd.DataFrame({
                            "neuron": kept_uniq_unit_List,
                            "value": patterns[ass]
                        })
                        x = np.arange(len(df))
                        y = df["value"].values                        
                        axes[ass].vlines(x, ymin=0, ymax=y, linewidth=2, color='k')
                        axes[ass].scatter(x, y, s=50, zorder=3, edgecolors=None, color='k')
                        axes[ass].axhline(y=thresh, color='r', linestyle='--')
                        axes[ass].axhline(y=-thresh, color='r', linestyle='--')
                        axes[ass].axhline(y=0, color='k')
                        axes[ass].set_xticks(x)
                        axes[ass].set_xticklabels(df['neuron'], rotation=45, fontsize=7, ha='right')
                        axes[ass].set_ylabel("Value")
                        axes[ass].set_title(f"{session_date}, {session_time}, {mice}, {ass}")                    
                    for i in range(n_ass, len(axes)):
                        axes[i].set_visible(False)                    
                    plt.tight_layout()
                    plt.savefig("//10.69.168.1/crnldata/forgetting/Théa/MiniscopeOE_analysis/plot_cell_weights.svg", bbox_inches="tight")
                    plt.show()
 
 
                    # -----------------------------------------------------------------------
                    # Plot calcium traces grouped by cell assembly (z-score, one line/neuron)
                    # -----------------------------------------------------------------------
 
                    def _zscore(trace):
                        m, s = np.mean(trace), np.std(trace)
                        return (trace - m) / s if s > 0 else np.zeros_like(trace)
 
                    # Isolate assemblies detected in THIS session only
                    sess_assemblies = {
                        k: v for k, v in CellAssembly_members[mice].items()
                        if k.startswith(session_time)
                    }
 
                    # Color palettes (dark = trace color, light = background shading)
                    _palette_dark = [
                        "#3C3489", "#0F6E56", "#993C1D", "#185FA5",
                        "#639922", "#BA7517", "#993556", "#A32D2D",
                    ]
                    _palette_light = [
                        "#AFA9EC", "#5DCAA5", "#F0997B", "#85B7EB",
                        "#97C459", "#EF9F27", "#ED93B1", "#F09595",
                    ]
                    _ass_ids = list(sess_assemblies.keys())
                    _asm_color  = {aid: _palette_dark[i % len(_palette_dark)]  for i, aid in enumerate(_ass_ids)}
                    _asm_light  = {aid: _palette_light[i % len(_palette_light)] for i, aid in enumerate(_ass_ids)}
 
                    # Map each unit to its assembly
                    _unit_to_asm = {}
                    for aid, members in sess_assemblies.items():
                        for uid in members:
                            _unit_to_asm[uid] = aid
 
                    # Build ordered neuron list: grouped by assembly, then unassigned
                    _ordered = []
                    _group_spans = {}   # aid -> (first_row, last_row)
                    for aid in _ass_ids:
                        _members_here = [u for u in sess_assemblies[aid] if u in kept_uniq_unit_List]
                        if _members_here:
                            _group_spans[aid] = (len(_ordered), len(_ordered) + len(_members_here) - 1)
                            _ordered.extend(_members_here)
                    _unassigned = [u for u in kept_uniq_unit_List if u not in _unit_to_asm]
                    if _unassigned:
                        _group_spans["unassigned"] = (len(_ordered), len(_ordered) + len(_unassigned) - 1)
                        _ordered.extend(_unassigned)
 
                    _n = len(_ordered)
                    _spacing = 3.0   # z-score units between traces
                    _ts = np.array(Calcium.columns, dtype=float)
 
                    _fig_h = max(4, 0.35 * _n + 2)
                    _fig, _ax = plt.subplots(figsize=(16, _fig_h))
 
                    # Assembly background shading
                    for aid, (fi, li) in _group_spans.items():
                        if aid == "unassigned":
                            continue
                        _y_bot = (_n - 1 - li) * _spacing - _spacing * 0.45
                        _y_h   = (li - fi + 1) * _spacing
                        _ax.axhspan(_y_bot, _y_bot + _y_h,
                                    alpha=0.07, color=_asm_light.get(aid, "#D3D1C7"), zorder=0)
 
                    # Draw one z-scored trace per neuron
                    for _row, uid in enumerate(_ordered):
                        if uid not in kept_uniq_unit_List:
                            continue
                        _li = kept_uniq_unit_List.index(uid)
                        _trace = _zscore(np.array(Calcium.iloc[_li], dtype=float))
                        _yoff  = (_n - 1 - _row) * _spacing
                        _col   = _asm_color.get(_unit_to_asm.get(uid, "unassigned"), "#888780")
                        _ax.plot(_ts, _trace + _yoff, color=_col, linewidth=0.7, alpha=0.85)
 
                    # Y-axis: neuron labels colored by assembly
                    _ytick_pos, _ytick_lab, _ytick_col = [], [], []
                    for _row, uid in enumerate(_ordered):
                        _ytick_pos.append((_n - 1 - _row) * _spacing)
                        _ytick_lab.append(uid)
                        _ytick_col.append(_asm_color.get(_unit_to_asm.get(uid, "unassigned"), "#888780"))
                    _ax.set_yticks(_ytick_pos)
                    _ax.set_yticklabels(_ytick_lab, fontsize=7)
                    for _tick, _col in zip(_ax.get_yticklabels(), _ytick_col):
                        _tick.set_color(_col)
 
                    # Assembly labels on right axis
                    _ax2 = _ax.twinx()
                    _ax2.set_ylim(_ax.get_ylim())
                    _ax2.set_yticks([])
                    for aid, (fi, li) in _group_spans.items():
                        if aid == "unassigned":
                            continue
                        _ymid = ((_n - 1 - fi) + (_n - 1 - li)) / 2 * _spacing
                        _short = aid.split("_CellAss")[-1] if "_CellAss" in aid else aid
                        _ax2.annotate(
                            f"Asm {_short}",
                            xy=(1.01, _ymid),
                            xycoords=("axes fraction", "data"),
                            fontsize=7, fontweight="bold",
                            color=_asm_color.get(aid, "#888780"),
                            va="center",
                        )
 
                    # Scale bar: 5 s / 1 z-score
                    _t0, _t1 = _ts[0], _ts[-1]
                    _sx_end   = _t1 - (_t1 - _t0) * 0.02
                    _sx_start = _sx_end - 5.0
                    _sy       = -_spacing * 0.8
                    _ax.plot([_sx_start, _sx_end], [_sy, _sy],       color='k', linewidth=1.5)
                    _ax.plot([_sx_end,   _sx_end ], [_sy, _sy + 1.0], color='k', linewidth=1.5)
                    _ax.text(_sx_end + 0.3, _sy + 0.5, "1 z-score", fontsize=7, va="center")
                    _ax.text((_sx_start + _sx_end) / 2, _sy - 0.7, "5 s", fontsize=7, ha="center")
 
                    _ax.set_xlim(_t0, _t1)
                    _ax.set_xlabel("Time (s)", fontsize=9)
                    _ax.set_title(f"{session_date}  |  {session_time}  |  {mice}", fontsize=10)
                    _ax.spines["top"].set_visible(False)
                    _ax.spines["right"].set_visible(False)
                    _ax2.spines["top"].set_visible(False)
                    _ax2.spines["right"].set_visible(False)
 
                    plt.tight_layout()
                    plt.savefig(
                        f"//10.69.168.1/crnldata/forgetting/Théa/MiniscopeOE_analysis/"
                        f"plot_calcium_by_assembly_{session_date}_{session_time}_{mice}.svg",
                        bbox_inches="tight"
                    )
                    plt.show()
 
                    # -----------------------------------------------------------------------
                    # End of calcium-by-assembly plot
                    # -----------------------------------------------------------------------
 
 
                    print('______________________________________________________________________________________________________________________________________________________________________________________________________________________________')


Calcic activity of selected cell from cell assemblies during sleep session

In [ ]:
DrugExperiment=0 # =1 if CGP Experiment // DrugExperiment=0 if Baseline Experiment
 
AnalysisID='' 
 
saveexcel=0
 
local = True
if local:
    dir = "//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_data/L1NDNF_mice/"
else: 
    dir = "/mnt/data/AurelieB_other/"
 
 
mapp = {1: 'AW',  2: 'QW', 3: 'NREM',  4: 'IS', 5: 'REM',  6: 'undefined'}
drugs=['baseline']
 
CellAssembly_dict={}
CellAssembly_members={}
for dpath in Path(dir).glob('**/mappingsAB.pkl'):
 
    mappfile = open(dpath.parents[0]/ f'mappingsAB.pkl', 'rb')
    mapping = pickle.load(mappfile)
    mapping_sess = mapping['session']   
 
    centroidfile = open(dpath.parents[0]/ f'centsAB.pkl', 'rb')
    centroids = pickle.load(centroidfile)
    centroids_sess = centroids['session']   
 
    mice = dpath.parents[1].parts[-1]
    NeuronType = dpath.parents[2].parts[-1]
    
    print(f"####################################################################################")
    print(f"################################### {mice} ####################################")
    print(f"####################################################################################")
 
    nb_minian_total=0
    dict_Path={}
    dict_Calcium = {}
    dict_Deconv = {}
    dict_Scoring = {}
    dict_Stamps = {}
    dict_TodropFile = {}
    dict_StampsMiniscope = {}
 
    CellAssembly_patterns=pd.DataFrame()  
    CellAssembly_members[mice]={}
 
    previousEndTime=0
    InitialStartTime=0
 
    assembly_nb=0
 
    minian_folders = [f for f in dpath.parents[0].rglob('minian') if f.is_dir()]
 
    for minianpath in minian_folders: # for each minian folders found in this mouse 
 
        if 1==1: # any(p in all_expe_types for p in minianpath.parts): # have to be to the expe_types
 
            if any(minianpath.parents[1].glob('*V4_Miniscope/timeStamps.csv')): 
                session_path=minianpath.parents[1]        
                tsmini= pd.read_csv(list(session_path.glob('*V4_Miniscope/timeStamps.csv'))[0])['Time Stamp (ms)']
                V4subfolder=False               
                if any(session_path.parent.rglob('OpenEphys/EphyViewer_ManualScor5s_6Stages.csv')):
                    session_type=minianpath.parents[3].name
                    session_date=minianpath.parents[4].name
                    session_time=minianpath.parents[1].name
                else:
                    session_type=minianpath.parents[2].name
                    session_date=minianpath.parents[3].name
                    session_time=minianpath.parents[1].name
                session=session_time        
            elif any(minianpath.parents[2].glob('*V4_Miniscope/timeStamps.csv')):
                session_path=minianpath.parents[2]      
                tsmini= pd.read_csv(list(session_path.glob('*V4_Miniscope/timeStamps.csv'))[0])['Time Stamp (ms)']                      
                V4subfolder=True
                session_type=minianpath.parents[4].name
                session_date=minianpath.parents[5].name
                session_time=minianpath.parents[0].name
                session=session_time
 


            # =======================================================================
            # NOUVELLE CELLULE : tracés calciques SleepAfter pour neurones sélectionnés
            # =======================================================================
 
            if session_type == 'SleepAfter':
 
                print(f"Processing {session_type} session: {session} on the {session_date}, subfolders = {V4subfolder}")
 
                minian_ds = open_minian(minianpath)
                dict_Calcium[session] = minian_ds['C']
 
                try:
                    try:
                        with open(minianpath.parent / f'TodropFileAB.json', 'r') as f:
                            unit_to_drop = json.load(f)
                            dict_TodropFile[session] = unit_to_drop
                    except:
                        with open(minianpath / f'TodropFileAB.json', 'r') as f:
                            unit_to_drop = json.load(f)
                            dict_TodropFile[session] = unit_to_drop
                except:
                    print(' !!!! Minian session not validated !!!!')
                    continue
                print(session_path)
 
                # Start time & freq miniscope
                dict_StampsMiniscope[session] = pd.read_csv(list(session_path.glob('*V4_Miniscope/timeStamps.csv'))[0])
                tsmini = dict_StampsMiniscope[session]['Time Stamp (ms)']
                minian_freq = round(1 / np.mean(np.diff(np.array(tsmini) / 1000)))
 
                freqLFP = 1000
 
                # Remove bad units
                C = dict_Calcium[session]
                unit_to_drop = dict_TodropFile[session]
                C_sel = C.drop_sel(unit_id=unit_to_drop)
                CalciumSub = pd.DataFrame(C_sel, index=C_sel['unit_id'])
 
                # Build kept_uniq_unit_List for this SleepAfter session
                indexMappList = mapping_sess[session]
                kept_uniq_unit_List_sleep = []
                for unit in CalciumSub.index:
                    indexMapp = np.where(indexMappList == unit)[0]
                    kept_uniq_unit_List_sleep.append(f"{mice}{str(indexMapp).replace('[','').replace(']','')}")
 
                if len(kept_uniq_unit_List_sleep) == 0:
                    print(f'no cells kept in the SleepAfter session: {session}')
                    continue
 
                # Realign traces to recorded timestamps
                firstframe_sleep = 0
                timestamps_sleep = np.array(tsmini[firstframe_sleep:firstframe_sleep + len(CalciumSub.T)]) / freqLFP
                new_timestamps_sleep = np.arange(timestamps_sleep[0], timestamps_sleep[-1], 1 / minian_freq)
 
                Calcium_sleep = pd.DataFrame(index=CalciumSub.index, columns=new_timestamps_sleep)
                for feature in CalciumSub.index:
                    interpolator = interpolate.interp1d(timestamps_sleep, CalciumSub.loc[feature], kind='linear')
                    Calcium_sleep.loc[feature] = interpolator(new_timestamps_sleep)
 
                print(f"SleepAfter session loaded — {len(kept_uniq_unit_List_sleep)} neurons available: {kept_uniq_unit_List_sleep}")
 
                # -----------------------------------------------------------------------
                # Neurons of interest: N5, N1, N22, NB1543, NB1545 (if present)
                # -----------------------------------------------------------------------
                neurons_of_interest = [
                    f"{mice}5",
                    f"{mice}1",
                    f"{mice}22",
                    f"{mice}1543",
                    f"{mice}1545",
                ]
                # Keep only those actually present in this session
                neurons_to_plot = [u for u in neurons_of_interest if u in kept_uniq_unit_List_sleep]
 
                if len(neurons_to_plot) == 0:
                    print(f"None of the target neurons found in SleepAfter session {session}. Available: {kept_uniq_unit_List_sleep}")
                    continue
 
                print(f"Neurons found and to be plotted: {neurons_to_plot}")
 
                # -----------------------------------------------------------------------
                # Z-score helper
                # -----------------------------------------------------------------------
                def _zscore_sleep(trace):
                    m, s = np.mean(trace), np.std(trace)
                    return (trace - m) / s if s > 0 else np.zeros_like(trace)
 
                # -----------------------------------------------------------------------
                # Plot: one z-scored trace per neuron, all in #0F6E56
                # -----------------------------------------------------------------------
                _spacing  = 3.0   # z-score units between traces
                _ts       = np.array(Calcium_sleep.columns, dtype=float)
                _n        = len(neurons_to_plot)
                _TRACE_COLOR = "#0F6E56"
 
                _fig_h = max(4, 0.35 * _n + 2)
                _fig, _ax = plt.subplots(figsize=(16, _fig_h))
 
                for _row, uid in enumerate(neurons_to_plot):
                    _li    = kept_uniq_unit_List_sleep.index(uid)
                    _trace = _zscore_sleep(np.array(Calcium_sleep.iloc[_li], dtype=float))
                    _yoff  = (_n - 1 - _row) * _spacing
                    _ax.plot(_ts, _trace + _yoff, color=_TRACE_COLOR, linewidth=0.7, alpha=0.85)
 
                # Y-axis: neuron labels
                _ytick_pos = [(_n - 1 - _row) * _spacing for _row in range(_n)]
                _ax.set_yticks(_ytick_pos)
                _ax.set_yticklabels(neurons_to_plot, fontsize=8, color=_TRACE_COLOR)
 
                # Scale bar: 5 s / 1 z-score
                _t0, _t1  = _ts[0], _ts[-1]
                _sx_end   = _t1 - (_t1 - _t0) * 0.02
                _sx_start = _sx_end - 5.0
                _sy       = -_spacing * 0.8
                _ax.plot([_sx_start, _sx_end], [_sy, _sy],        color='k', linewidth=1.5)
                _ax.plot([_sx_end,   _sx_end ], [_sy, _sy + 1.0], color='k', linewidth=1.5)
                _ax.text(_sx_end + 0.3, _sy + 0.5, "1 z-score", fontsize=7, va="center")
                _ax.text((_sx_start + _sx_end) / 2, _sy - 0.7, "5 s", fontsize=7, ha="center")
 
                _ax.set_xlim(_t0, _t1)
                _ax.set_xlabel("Time (s)", fontsize=9)
                _ax.set_title(f"{session_date}  |  {session_time}  |  {mice}  |  SleepAfter", fontsize=10)
                _ax.spines["top"].set_visible(False)
                _ax.spines["right"].set_visible(False)
                _ax.spines["left"].set_visible(False)
                _ax.set_ylabel("")
 
                plt.tight_layout()
                plt.savefig(
                    f"//10.69.168.1/crnldata/forgetting/Théa/MiniscopeOE_analysis/"
                    f"plot_calcium_SleepAfter_{session_date}_{session_time}_{mice}.svg",
                    bbox_inches="tight"
                )
                plt.show()
 
                print(f"SleepAfter calcium plot saved for {mice}, session {session_time}.")
                print('______________________________________________________________________________________________________________________________________________________________________________________________________________________________')
